In [ ]:
import os, glob, re, string, pickle
import pandas as pd
import numpy as np
from numpy.linalg import norm
from gensim.models import KeyedVectors

# 0. LOAD FASTTEXT EMBEDDINGS ONCE
print("Loading FastText embeddings…")
ft_model = KeyedVectors.load_word2vec_format("/Users/harshit/Desktop/cc.en.300.vec", binary=False)
print("FastText loaded:", len(ft_model.key_to_index), "words,", ft_model.vector_size, "dims")

In [ ]:
# 1. PATH CONFIGURATION
base_data = "/Users/harshit/Desktop/KelloggXParlamint/Kellogg/data"
country     = "FI" #Change as per country
corpus_dir  = os.path.join(base_data, "raw", f"ParlaMint-{country}-en.txt")
embed_path = os.path.join(base_data, "embeddings", "cc.en.300.vec")
results_dir = os.path.join(base_data, "results","transcript", country)
os.makedirs(results_dir, exist_ok=True)

In [ ]:
# 3. LOAD & CLEAN PARLAMINT-FR EN SPEECHES
#    Reads all .txt under ParlaMint-FR/[year]/ → DataFrame with columns 'year' & 'clean_text'
speech_files = glob.glob(os.path.join(corpus_dir, "*/*.txt"))
texts, years = [], []
for path in speech_files:
    years.append(os.path.basename(os.path.dirname(path)))
    with open(path, 'r', encoding='utf-8') as f:
        texts.append(f.read())

df = pd.DataFrame({"year": years, "text": texts})
# Clean text: lowercase, strip punctuation, collapse whitespace
# Cast to str to avoid non-string entries
df['text'] = df['text'].astype(str)
df['clean_text'] = (
    df['text']
      .str.lower()
      .str.replace(f"[{re.escape(string.punctuation)}]", " ", regex=True)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)
print("Loaded & cleaned speeches:", df.shape, "rows")

In [ ]:
# 4. DEFINE WORD GROUPS
nature_terms     = ["nature","climate","environment","land","forest","forests","biodiversity","restoration","reforestation","ecology"]
importance_terms = ["important","importance","significant","meaningful"]
window = 5
years_sorted = sorted(df['year'].unique())

In [ ]:
# 5. EXTRACT CONTEXT WINDOWS PER (TERM, YEAR)
#    contexts_by_term_year[term][year] = list of token lists
contexts_by_term_year = {
    term: {yr: [] for yr in years_sorted}
    for term in nature_terms
}

for _, row in df.iterrows():
    yr = row['year']
    tokens = row['clean_text'].split()
    for i, tok in enumerate(tokens):
        if tok in nature_terms:
            start = max(0, i-window)
            end   = min(len(tokens), i+window+1)
            ctx = tokens[start:i] + tokens[i+1:end]  # exclude target
            contexts_by_term_year[tok][yr].append(ctx)

# SAVE contexts_by_term_year for reuse
import pickle
with open(os.path.join(results_dir, "contexts_by_term_year.pkl"), "wb") as f:
    pickle.dump(contexts_by_term_year, f)
print("Saved contexts_by_term_year → contexts_by_term_year.pkl")

# SAVE a flat CSV
rows = [
    {"year": yr, "term": term, "context": " ".join(ctx)}
    for term, yd in contexts_by_term_year.items()
    for yr, ctxs in yd.items()
    for ctx in ctxs
]
pd.DataFrame(rows).to_csv(os.path.join(results_dir, "contexts_by_term_year.csv"), index=False)
print("Saved contexts_by_term_year CSV → contexts_by_term_year.csv")

In [ ]:
# 6. COMPUTE ALC EMBEDDINGS per (nature_term, year)
alc_embeddings = {term: {} for term in nature_terms}
for term, year_dict in contexts_by_term_year.items():
    for yr, ctxs in year_dict.items():
        inst_vecs = []
        for ctx in ctxs:
            vs = [ft_model[w] for w in ctx if w in ft_model.key_to_index]
            if vs:
                inst_vecs.append(np.mean(vs, axis=0))
        if inst_vecs:
            alc_embeddings[term][yr] = np.mean(inst_vecs, axis=0)
print("ALC embeddings computed for all terms and years")

# SAVE alc_embeddings
with open(os.path.join(results_dir, "alc_embeddings.pkl"), "wb") as f:
    pickle.dump(alc_embeddings, f)
print("Saved alc_embeddings → alc_embeddings.pkl")


In [ ]:
# 7. LOAD & SAVE STATIC IMPORTANCE EMBEDDINGS
imp_embeddings = {
    imp: ft_model[imp]
    for imp in importance_terms
    if imp in ft_model.key_to_index
}
with open(os.path.join(results_dir, "imp_embeddings.pkl"), "wb") as f:
    pickle.dump(imp_embeddings, f)
print("Saved imp_embeddings → imp_embeddings.pkl")


# 8. BUILD & SAVE 10×4 COSINE-TABLES PER YEAR
for yr in years_sorted:
    data = {}
    for term in nature_terms:
        if yr in alc_embeddings[term]:
            vec_n = alc_embeddings[term][yr]
            sims = {
                imp: float(np.dot(vec_n, vec_i)/(norm(vec_n)*norm(vec_i)))
                for imp, vec_i in imp_embeddings.items()
            }
            data[term] = sims
    df_year = pd.DataFrame.from_dict(data, orient='index') \
                        .reindex(nature_terms)
    out_csv = os.path.join(results_dir, f"{yr}_nature_vs_importance.csv")
    df_year.to_csv(out_csv, index=True)  
    print(f"Saved {country}/{yr}_nature_vs_importance.csv")

print("All done!")